In [ ]:
import os
import anndata as ad
import numpy as np
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import omicverse as ov
import scvi
from scvi.model.utils import mde

import warnings
warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

In [2]:
sc.settings.set_figure_params(dpi=100, frameon=False)
sc.set_figure_params(dpi=100)
sc.set_figure_params(figsize=(3, 3))
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (3, 3)

In [ ]:
# Change the working directory to the Garfield folder (if needed)
os.chdir('/storage2/liuxiaodongLab/fanxueying/embryo_benchmarking_rebuttal/code/20250812_label_transfer_integration_label_transfer_quantification')
os.getcwd()

In [4]:
# Process query datasets from folder
query_folder = '/storage2/liuxiaodongLab/fanxueying/mayanalysis/2024Aug/garfield/in_vitro_embryo_models/processed/data'
# List all h5ad files in the directory
h5ad_files = [os.path.join(query_folder, f) for f in os.listdir(query_folder) if f.startswith('corrected_processed_')]

In [ ]:
# Read each h5ad file into an AnnData object
adata_list = []
for file in h5ad_files:
    print(f"Reading file: {file}")
    adata = sc.read_h5ad(file)  # Read the h5ad file into AnnData
    adata_list.append(adata)  # Add AnnData object to the list

In [ ]:
# Define the output directory
output_dir = '/storage2/liuxiaodongLab/fanxueying/embryo_benchmarking_rebuttal/code/20250812_label_transfer_method_quantification_v3/embryo_model_garfield'

# Step 1: List all CSV files in the directory
garfield_files = [os.path.join(output_dir, f) for f in os.listdir(output_dir) if f.endswith('.csv')]

# Step 2: Initialize an empty dictionary to store the results
garfield_tables = {}

# Step 3: Process each file
for file in garfield_files:
    # Read the CSV file into a DataFrame
    garfield = pd.read_csv(file)
    
    # Extract the desired substring from the file name
    file_name = os.path.basename(file)  # Get the file name without the path
    new_name = file_name.replace('corrected_processed_', '')  # Remove "corrected_processed_"
    new_name = new_name.replace('.h5ad', '')  # Remove ".h5ad"
    new_name = new_name.split('_garfield')[0]  # Remove "_garfield" and anything after
    
    # Store the DataFrame in the dictionary with the new name
    garfield_tables[new_name] = garfield

# Now `garfield_tables` is a dictionary where keys are the processed file names and values are the DataFrames

# List of attributes to be extracted from Garfield results
attri = ["transferred_reanno_unfiltered", "transferred_reanno_uncert", 
         "transferred_lineage_unfiltered", "transferred_lineage_uncert"]

# Process each AnnData object and merge with Garfield results
for i, dataset in enumerate(adata_list):
    # Extract the dataset name from the file name (assuming `garfield_tables` keys are the file names without paths)
    name = os.path.basename(h5ad_files[i]).replace('.h5ad', '')  # Remove the file extension
    name = name.replace('corrected_processed_', '')  # Remove the file prefix
    
    # Get the corresponding Garfield results
    garf = garfield_tables.get(name)
    
    if garf is not None:
        # Check if 'X' is the first column or index
        if garf.columns[0] != 'X':  # If the first column is not 'X'
            print(f"Warning: The first column is not 'X' for {name}. Renaming the first column.")
            garf.rename(columns={garf.columns[0]: 'X'}, inplace=True)
        
        # Set the first column as index (row names)
        garf.set_index('X', inplace=True)
        
        # Extract the required columns from Garfield table
        garf = garf[attri]
        
        # Remove filtered cells from dataset
        dataset = dataset[dataset.obs_names.isin(garf.index), :]
        
        # Ensure row names in dataset and Garfield data match
        garf = garf.loc[dataset.obs_names, :]
        
        # Add prefix to Garfield column names
        garf.columns = [f"human_ref_{col}" for col in garf.columns]

        # Merge Garfield results into the AnnData metadata
        dataset.obs = pd.concat([dataset.obs, garf], axis=1)

        # Update the dataset in the list
        adata_list[i] = dataset
    else:
        print(f"No Garfield data found for {name}")


In [ ]:
# Merge all AnnData objects into one
# We use `anndata.concat()` to merge them, making sure the 'obs' and 'var' fields align
combined_adata = ad.concat(adata_list, label='batch', join='outer')

# Print the shape of the merged AnnData to verify
print(f"Combined AnnData shape: {combined_adata.shape}")


In [ ]:
combined_adata

In [ ]:
set(combined_adata.obs["orig.ident"])

In [ ]:
print(combined_adata.X)

In [11]:
combined_adata.layers["counts"] = combined_adata.X.copy()

In [12]:
sc.settings.seed = 42
sc.pp.normalize_total(combined_adata, target_sum=1e4)
sc.pp.log1p(combined_adata)
combined_adata.layers["logcounts"] = combined_adata.X.copy()
sc.pp.highly_variable_genes(combined_adata, n_top_genes=2000, flavor="cell_ranger", batch_key="orig.ident")
sc.tl.pca(combined_adata, n_comps=30, use_highly_variable=True)

In [13]:
adata_hvg = combined_adata[:, combined_adata.var.highly_variable].copy()

In [ ]:
adata_hvg

In [ ]:
####Unintegrated
combined_adata.obsm["Unintegrated"] = adata_hvg.obsm["X_pca"]
adata_hvg.obsm["Unintegrated"] = adata_hvg.obsm["X_pca"]
sc.pp.neighbors(combined_adata, use_rep="Unintegrated",random_state=42)
sc.tl.leiden(combined_adata, resolution=0.5,key_added=f"Unintegrated_res_0.5",random_state=42)
sc.tl.umap(combined_adata,random_state=42)
combined_adata.obsm['X_Unintegrated'] = combined_adata.obsm['X_umap']
sc.pl.umap(combined_adata, color=['orig.ident'], save="Unintegrated_orig_ident.pdf")
sc.pl.umap(combined_adata, color=['stage'], save="Unintegrated_stage.pdf")
sc.pl.umap(combined_adata, color=['human_ref_transferred_lineage_unfiltered'], save="Unintegrated_transferred_lineage.pdf")

In [ ]:
###scVI
scvi.model.SCVI.setup_anndata(adata_hvg, layer="counts", batch_key="orig.ident")
vae = scvi.model.SCVI(adata_hvg, gene_likelihood="nb", n_layers=2, n_latent=30)
vae.train()

In [17]:
combined_adata.obsm["scVI"] = vae.get_latent_representation()

In [ ]:
###scANVI
lvae = scvi.model.SCANVI.from_scvi_model(
    vae,
    adata=adata_hvg,
    labels_key="human_ref_transferred_lineage_unfiltered",
    unlabeled_category="Unknown",
)
lvae.train(max_epochs=20, n_samples_per_label=100)

In [19]:
combined_adata.obsm["scANVI"] = lvae.get_latent_representation()

In [ ]:
adata_hvg.obsm["scANVI"] = combined_adata.obsm["scANVI"]
sc.pp.neighbors(combined_adata, use_rep="scANVI",random_state=42)
sc.tl.leiden(combined_adata, resolution=0.5,key_added=f"scANVI_res_0.5",random_state=42)
sc.tl.umap(combined_adata,random_state=42)
combined_adata.obsm['X_scANVI'] = combined_adata.obsm['X_umap']
adata_hvg.obsm['X_scANVI'] = combined_adata.obsm['X_scANVI']
sc.pl.umap(combined_adata, color=['orig.ident'],save="scANVI_orig.ident.pdf")
sc.pl.umap(combined_adata, color=['stage'],save="scANVI_stage.pdf")
sc.pl.umap(combined_adata, color=['human_ref_transferred_lineage_unfiltered'],save="scANVI_transferred_lineage.pdf")

In [21]:
# Check current working directory
print("Current working directory:", os.getcwd())

Current working directory: /storage2/liuxiaodongLab/fanxueying/embryo_benchmarking_rebuttal/code/20250812_label_transfer_integration_label_transfer_quantification


In [22]:
combined_adata.raw.var.rename(columns={'_index': 'index'}, inplace=True)
combined_adata.write_h5ad(filename="embryo_model_integration_garfield.h5ad")